[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/10_SequenceModels.ipynb#copy=true)

# Sequence Models: RNNs, LSTMs, and Model Comparison

**Module 0 · Lesson 10 of 13 · Student edition**  
**Estimated class time:** 90–110 minutes  
**Source sequence:** Original Day 3  

**Prerequisite:** Lessons 7–9  

## Learning objectives

By the end of this lesson, you should be able to:

- Explain hidden state, recurrence, and vanishing gradients.
- Implement and train RNN and LSTM forecasters.
- Compare classical, tree-based, MLP, RNN, and LSTM forecasts.

## Setup for this lesson

This cell recreates the data and completed prerequisites from earlier lessons, so this notebook can be run in a fresh kernel.

In [ ]:
# Shared forecasting setup from the preceding lesson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.ar_model import AutoReg
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff'] = df['Log_Passengers'].diff()
series = df['Log_Diff'].dropna().values

def make_lag_matrix(series, n_lags):
    series = np.asarray(series)
    X, y = [], []
    for t in range(n_lags, len(series)):
        X.append(series[t - n_lags:t])
        y.append(series[t])
    return np.asarray(X), np.asarray(y)

N_LAGS = 12
X, y = make_lag_matrix(series, N_LAGS)
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

naive_preds = np.concatenate([[y_train[-1]], y_test[:-1]])
train_series = series[:split + N_LAGS]
ar_result = AutoReg(train_series, lags=N_LAGS).fit()
ar_preds = ar_result.predict(
    start=len(train_series),
    end=len(train_series) + len(y_test) - 1,
    dynamic=False,
)


rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

gb_model = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, random_state=42
)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)

def compute_metrics(y_true, y_pred, label='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f'{label:<30}  RMSE={rmse:.5f}   MAE={mae:.5f}')
    return {'RMSE': rmse, 'MAE': mae}

results = {
    'Naive Baseline': compute_metrics(y_test, naive_preds, 'Naive Baseline'),
    'AR(12)': compute_metrics(y_test, ar_preds, 'AR(12)'),
    'Random Forest': compute_metrics(y_test, rf_preds, 'Random Forest'),
    'Gradient Boosting': compute_metrics(y_test, gb_preds, 'Gradient Boosting'),
}


scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)
y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

class MLPForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

mlp = MLPForecaster(input_size=N_LAGS, hidden_size=32)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
for _ in range(300):
    mlp.train()
    optimizer.zero_grad()
    loss = loss_fn(mlp(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

mlp.eval()
with torch.no_grad():
    mlp_preds_s = mlp(X_test_t).numpy()
mlp_preds = scaler_y.inverse_transform(mlp_preds_s.reshape(-1, 1)).ravel()
results['MLP'] = compute_metrics(y_test, mlp_preds, 'MLP')


***
## Part 6 — RNN & LSTM: Hidden State, Recurrence, and Gating

### Why sequence models?

The MLP receives a flat lag vector and treats lags as **unordered** features. A **Recurrent
Neural Network (RNN)** processes the lag sequence *one step at a time*, maintaining a
**hidden state** that is updated at each step. This allows the model to encode the
sequential structure of time explicitly.

### The RNN Recurrence Equation

At each time step $t$ within the sequence:

$$\mathbf{h}_t = \tanh(W_{hh}\,\mathbf{h}_{t-1} + W_{xh}\,\mathbf{x}_t + \mathbf{b}_h)$$

where:
- $\mathbf{x}_t$ — the input (here, the value of the series at lag $t$)
- $\mathbf{h}_t$ — the **hidden state**: a learned summary of the sequence so far
- $W_{hh}$ — **recurrent weights** connecting consecutive hidden states
- $W_{xh}$ — **input weights**
- $\mathbf{h}_0 = \mathbf{0}$ (initialized to zero)

After processing all $p$ lags, the final hidden state $\mathbf{h}_p$ is passed to an
output layer:

$$\hat{y} = W_o \mathbf{h}_p + b_o$$

### The Vanishing Gradient Problem

During training, gradients must flow backward through the recurrence. If the recurrent
weights are small, the gradient shrinks exponentially over time steps — this is the
**vanishing gradient problem**, which prevents RNNs from learning long-range dependencies.

### The LSTM Solution

The **LSTM (Long Short-Term Memory)** introduces a **cell state** $\mathbf{c}_t$ and
three **gates** that control what information is retained, discarded, and output:

$$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate — what to erase from cell state)}$$
$$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate — what new info to write)}$$
$$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate — what to expose as hidden state)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(cell state update)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(hidden state output)}$$

The cell state $\mathbf{c}_t$ acts as a "conveyor belt" — it can carry information
across many steps without vanishing, because the **forget gate allows additive gradient
flow** instead of multiplicative.

### 6.1 Implement the RNN

**Your turn!** Complete the `RNNForecaster` class below.

> 💡 **PyTorch note:** `nn.RNN(input_size, hidden_size, batch_first=True)` processes a
> sequence tensor of shape `(batch, seq_len, input_size)`. The output `(output, h_n)` gives
> all hidden states and the final hidden state. We unsqueeze the lag vector to add the
> `input_size=1` dimension: `x.unsqueeze(-1)` → `(batch, n_lags, 1)`.

In [ ]:
class RNNForecaster(nn.Module):
    """
    Single-layer Elman RNN for time series forecasting.

    Processes the lag sequence step-by-step and produces a single
    scalar forecast from the final hidden state.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden state dimension
    """
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # FILL IN: create an nn.RNN layer with batch_first=True
        self.rnn = nn.RNN(???, ???, batch_first=True)
        # FILL IN: output layer mapping hidden_size → 1
        self.fc  = nn.Linear(???, ???)

    def forward(self, x):
        # x shape: (batch, n_lags) → add feature dimension
        x = x.unsqueeze(-1)           # (batch, n_lags, 1)
        # FILL IN: pass through RNN, extract final hidden state h_n
        _, h_n = self.rnn(???)
        h_n = h_n.squeeze(0)          # (batch, hidden)
        return self.fc(h_n).squeeze(-1)

### 6.2 Implement the LSTM

> 💡 **Note:** `nn.LSTM` returns `(output, (h_n, c_n))` — the tuple includes both the
> hidden state and the cell state. We use only `h_n` for the output prediction.

In [ ]:
class LSTMForecaster(nn.Module):
    """
    Single-layer LSTM for time series forecasting.

    Extends the RNN with a cell state and gating mechanisms, allowing
    the model to selectively retain or forget information across lags.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden/cell state dimension
    """
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # FILL IN: create an nn.LSTM layer with batch_first=True
        self.lstm = nn.LSTM(???, ???, batch_first=True)
        # FILL IN: output layer mapping hidden_size → 1
        self.fc   = nn.Linear(???, ???)

    def forward(self, x):
        x = x.unsqueeze(-1)               # (batch, n_lags, 1)
        # FILL IN: pass through LSTM, extract h_n from the returned tuple
        _, (h_n, _) = self.lstm(???)
        h_n = h_n.squeeze(0)              # (batch, hidden)
        return self.fc(h_n).squeeze(-1)

### 6.3 Train the RNN and LSTM

The function below trains any PyTorch model with MSE loss. Complete it, then train both
the RNN and LSTM.

### 🤝 Dividir y Confluir

Your instructor will assign your group a `hidden_size` to experiment with:

| Group | Model | hidden_size |
|---|---|---|
| A | RNN | 16 |
| B | RNN | 64 |
| C | LSTM | 16 |
| D | LSTM | 64 |

Fit your assigned configuration, compute RMSE/MAE, and share results for class comparison.

In [ ]:
def train_model(model, X_t, y_t, epochs=300, lr=1e-3, label='Model'):
    """
    Train a PyTorch model with MSE loss and Adam optimizer.

    Parameters
    ----------
    model  : nn.Module
    X_t    : torch.Tensor, (n_samples, n_lags)
    y_t    : torch.Tensor, (n_samples,)
    epochs : int
    lr     : float, learning rate
    label  : str, for display

    Returns
    -------
    list of training losses
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    losses    = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        # FILL IN: forward pass and loss
        pred = model(???)
        loss = loss_fn(???, y_t)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())

    print(f'{label} — final loss: {losses[-1]:.6f}')
    return losses


rnn_model  = RNNForecaster(hidden_size=32)
lstm_model = LSTMForecaster(hidden_size=32)

rnn_losses  = train_model(rnn_model,  X_train_t, y_train_t, epochs=300, label='RNN')
lstm_losses = train_model(lstm_model, X_train_t, y_train_t, epochs=300, label='LSTM')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(rnn_losses,  label='RNN',  color='steelblue')
ax.plot(lstm_losses, label='LSTM', color='crimson')
ax.set_title('Training Loss — RNN vs LSTM')
ax.set_xlabel('Epoch')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def eval_model(model, X_t, scaler_y):
    """
    Generate predictions and inverse-transform to original scale.

    Parameters
    ----------
    model    : trained nn.Module
    X_t      : torch.Tensor, test features
    scaler_y : fitted MinMaxScaler for the target

    Returns
    -------
    np.ndarray of predictions in original (log-diff) scale
    """
    model.eval()
    with torch.no_grad():
        # FILL IN: run model on test features
        preds_s = model(???).numpy()
    # FILL IN: inverse-transform
    return scaler_y.inverse_transform(preds_s.reshape(-1, 1)).ravel()


rnn_preds  = eval_model(rnn_model,  X_test_t, scaler_y)
lstm_preds = eval_model(lstm_model, X_test_t, scaler_y)

results['RNN']  = compute_metrics(y_test, rnn_preds,  'RNN')
results['LSTM'] = compute_metrics(y_test, lstm_preds, 'LSTM')

### ✏️ Written Response 6

Answer each question in 2–4 sentences:

1. What is the **hidden state** in an RNN, and what role does it play that the MLP's
   lag vector does not?
2. What is the **vanishing gradient problem** in plain language? Why does it make long-range
   dependencies hard to learn?
3. Pick one LSTM gate (forget, input, or output). Describe in your own words what it does
   and why it helps compared to a plain RNN.
4. On this dataset, did the LSTM outperform the RNN by a large margin? Why or why not?

> **YOUR ANSWER:**

***
## Part 7 — Full Model Comparison

### 7.1 Build the comparison table

**Your turn!** Assemble all results into a sorted DataFrame.

In [ ]:
# FILL IN: create a DataFrame from the results dict and sort by RMSE
comparison_df = pd.DataFrame(???).T
comparison_df = comparison_df.sort_values(???)

print('\n=== Model Comparison (sorted by RMSE) ===')
print(comparison_df.to_string(float_format='{:.5f}'.format))

In [ ]:
# Visualise all predictions
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
test_range = np.arange(len(y_test))

# Top panel: classical + tree models
axes[0].plot(test_range, y_test,          label='Actual',            color='black',      linewidth=2)
axes[0].plot(test_range, naive_preds,     label='Naive',             color='gray',       linestyle='--')
axes[0].plot(test_range, ar_preds.values, label='AR(12)',            color='steelblue',  linestyle='-.')
axes[0].plot(test_range, rf_preds,        label='Random Forest',     color='forestgreen')
axes[0].plot(test_range, gb_preds,        label='Gradient Boosting', color='darkorange')
axes[0].set_title('Classical & Tree-Based Models')
axes[0].legend(fontsize=9)

# Bottom panel: neural models
axes[1].plot(test_range, y_test,          label='Actual', color='black',  linewidth=2)
axes[1].plot(test_range, mlp_preds,       label='MLP',    color='purple')
axes[1].plot(test_range, rnn_preds,       label='RNN',    color='teal')
axes[1].plot(test_range, lstm_preds,      label='LSTM',   color='crimson')
axes[1].set_title('Neural Models')
axes[1].set_xlabel('Test Step')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 4))
models    = list(comparison_df.index)
rmse_vals = comparison_df['RMSE'].values
mae_vals  = comparison_df['MAE'].values

x = np.arange(len(models))
w = 0.35
ax.bar(x - w/2, rmse_vals, w, label='RMSE', color='steelblue', alpha=0.8)
ax.bar(x + w/2, mae_vals,  w, label='MAE',  color='crimson',   alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set_title('Model Comparison: RMSE and MAE on Test Set')
ax.legend()
plt.tight_layout()
plt.show()

### ✏️ Final Written Reflection

Write a **4–6 sentence summary** as if reporting to a colleague who did not attend today.
Your summary must address:

- Why temporal ordering is critical for train/test splits in time series
- What the naive baseline tells you and why it matters
- The key algorithmic difference between the MLP and the RNN/LSTM
- How gating in the LSTM addresses the vanishing gradient problem
- Which model performed best on this dataset and one hypothesis for why

**Share your summary with the class (dividir y confluir debrief).**

> **YOUR ANSWER:**

---

## Moving Forward

Choose a different publicly available time series (e.g., daily stock prices, electricity
consumption, or temperature data) and run the complete pipeline from today:

1. Apply appropriate transformations to achieve stationarity
2. Build a lag-embedded feature matrix with a chosen `n_lags`
3. Create a time-aware train/test split
4. Train at least three of the six model families from today
5. Report RMSE and MAE for each model and interpret the results
6. Write 3–5 sentences comparing your findings to the Air Passengers results

```python
# Your code here
```

---

## References / Further Reading

* [Forecasting: Principles and Practice — Chapter 5 (Regression models)](https://otexts.com/fpp3/regression.html)
* [Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow — Chapter 15 (RNNs)](https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/)
* [Understanding LSTM Networks — Colah's Blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)
* [PyTorch `nn.RNN` documentation](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)
* [PyTorch `nn.LSTM` documentation](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* [Scikit-Learn `RandomForestRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)